In [ ]:
%reload_ext autoreload
%autoreload 2

import os
from pathlib import Path

print(Path().cwd())
os.chdir(Path(os.getcwd()).parent)
print(Path().cwd())

## Select Contrast-Enhanced Ultrasound (CEUS) Cine and Parser

In [ ]:
from src.image_loading.options import get_scan_loaders

print("Available scan loaders:", list(get_scan_loaders().keys()))

In [ ]:
scan_type = 'nifti'

# Takes the DICOM file as input for contrast enhanced ultrasound (CEUS) scans
CEUS_scan_path = '/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/SIP/HighQualityData/UCSD/P07/V03/UCSD-P07-V03-CE1_10.46.36_mf_sip_capture_50_2_1_0_CEUS.nii'
bmode_scan_path = '/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/SIP/HighQualityData/UCSD/P07/V03/UCSD-P07-V03-CE1_10.46.36_mf_sip_capture_50_2_1_0_BMODE.nii'
scan_loader_kwargs = {
}

In [ ]:
from src.entrypoints import scan_loading_step

image_data = scan_loading_step(scan_type, CEUS_scan_path, **scan_loader_kwargs)
bmode_image_data = scan_loading_step(scan_type, bmode_scan_path, **scan_loader_kwargs)

## Load Segmentation

Assumes same segmentation for each frame

In [ ]:
from src.seg_loading.options import get_seg_loaders

print("Available segmentation loaders:", list(get_seg_loaders().keys()))

In [ ]:
seg_type = 'nifti'

seg_path = '/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/SIP/HighQualityData/UCSD/P07/V03/UCSD-P07-V03-CE1-MC_VOI.nii.gz'
seg_loader_kwargs = {}

In [ ]:
from src.entrypoints import seg_loading_step

# Testing the motion compensation, right now is hard coded
seg_data = seg_loading_step(seg_type, image_data, seg_path, CEUS_scan_path, **seg_loader_kwargs)

# Figure 1 Display the motion compensation from B mode and CEUS


Figure 1: Axial Plane of the 3D contrast enhanced ultrasound and B mode in 5 consecutive frames. Top row: no motion compensation. Bottom row: motion compensated data. The red dash line represents the boarder of the region of interests


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import binary_erosion

def get_mask_boundary(mask_slice):
    if mask_slice.max() == 0:
        return np.zeros_like(mask_slice, dtype=bool)
    eroded = binary_erosion(mask_slice)
    return mask_slice.astype(bool) & ~eroded

def get_voi_center(mask_3d):
    coords = np.where(mask_3d > 0)
    if len(coords[0]) == 0:
        return None, None, None
    return (int(np.mean(coords[0])),   # lateral  X
            int(np.mean(coords[1])),   # depth    Y
            int(np.mean(coords[2])))   # elevation Z
def enhance_bmode_noise(image_slice, p_low_percentile=15.0, p_high_percentile=98.5):
    non_zero = image_slice[image_slice != 0]
    p_low = np.percentile(non_zero, p_low_percentile)
    p_high = np.percentile(non_zero, p_high_percentile)
    clipped = np.clip(image_slice, p_low, p_high)
    return ((clipped - p_low) / (p_high - p_low) * 255).astype(np.uint8)

def apply_clahe(img_u8, clip=2.0, grid=8):
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(grid, grid))
    return clahe.apply(img_u8)

In [ ]:
from scipy.ndimage import shift as ndshift

# --- voxel spacing (mm). extras_dict stores it as (z, y, x) ---
# volume axes are (X=lateral, Y=depth, Z=elevation, T)
sz, sy, sx = image_data.pixdim   # z, y, x  in mm

nx, ny, nz, num_frames = image_data.pixel_data.shape

show_frames = [20, 21, 22, 23, 24]
ticks = [0, 20, 40, 60, 80]

mask = seg_data.seg_mask                                  # fixed reference VOI
lat_c, dep_c, ele_c = get_voi_center(mask)
bnd = get_mask_boundary(np.transpose(mask[:, :, ele_c]))  # same contour everywhere
roi_top_mm = np.argwhere(np.transpose(mask[:, :, ele_c]).any(axis=1)).min() * sy
roi_bot_mm = np.argwhere(np.transpose(mask[:, :, ele_c]).any(axis=1)).max() * sy

mc = seg_data.motion_compensation
ext_axial = [0, nx * sx, ny * sy, 0]

fig, axes = plt.subplots(2, 5, figsize=(22, 9))

for col, frame in enumerate(show_frames):
    Vol = bmode_image_data.pixel_data[:, :, :, frame]

    # --- row 0: NON-COMPENSATED (raw volume) ---
    raw = np.transpose(Vol[:, :, ele_c]).astype(np.float64)
    img = enhance_bmode_noise(raw, p_low_percentile=5.0, p_high_percentile=99.5) 
    axes[0, col].imshow(img, cmap='gray', extent=ext_axial, aspect='equal')
    axes[0, col].axhline(roi_top_mm, color='red', ls='--', lw=1.5)
    axes[0, col].axhline(roi_bot_mm, color='red', ls='--', lw=1.5)

    # --- row 1: COMPENSATED (volume registered back) ---
    dx, dy, dz = mc.get_translation(frame)
    Vol_mc = ndshift(Vol, shift=[-dx, -dy, -dz], order=1, cval=0)
    raw_mc = np.transpose(Vol_mc[:, :, ele_c]).astype(np.float64)
    img_mc = enhance_bmode_noise(raw_mc, p_low_percentile=5.0, p_high_percentile=99.5)
    # img_mc = apply_clahe(img_mc.astype(np.uint8), clip=3, grid=8) 
    axes[1, col].imshow(img_mc, cmap='gray', extent=ext_axial, aspect='equal')
    axes[1, col].axhline(roi_top_mm, color='red', ls='--', lw=1.5)
    axes[1, col].axhline(roi_bot_mm, color='red', ls='--', lw=1.5)

# Apply tick marks to all axes
for ax in axes.flat:
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.tick_params(axis='both', labelsize=16)
    

# Only show tick labels on the bottom row (x) and left column (y)
for col in range(5):
    axes[0, col].set_xticklabels([])          # hide x labels on top row
    if col != 0:
        axes[0, col].set_yticklabels([])      # hide y labels except leftmost
        axes[1, col].set_yticklabels([])

plt.tight_layout()
plt.show()

In [ ]:
# CEUS images enhance with noise reduction from first frame
def compute_ceus_noise_floor(first_frame) -> float:
    """
    Compute the noise floor (p_low scalar) from pre-contrast frames of a 4D CEUS scan.
    Returns the noise floor value as a float.
    """
    pixel_data = first_frame

    ref_frames = pixel_data
    ref_nonzero = ref_frames[ref_frames != 0]
    if ref_nonzero.size == 0:
        raise ValueError("Pre-contrast reference frames contain no non-zero values.")

    noise_mean = np.mean(ref_nonzero)
    noise_std = np.std(ref_nonzero)
    return float(noise_mean+noise_std)  # noise floor is mean + std

def enhance_ceus(slice, p_low, p_high_percentile=99.5):
    """
    Enhance a single CEUS image slice using noise floor and high percentile.
    Returns the enhanced image slice.
    """
    non_zero = slice[slice != 0]
    if non_zero.size == 0:
        return np.zeros_like(slice, dtype=np.uint8)

    p_high = np.percentile(non_zero, p_high_percentile)
    clipped = np.clip(slice, p_low, p_high)
    return ((clipped - p_low) / (p_high - p_low) * 255).astype(np.uint8)

In [ ]:
show_frames = [20, 21, 22, 23, 24]

mask = seg_data.seg_mask                                  # fixed reference VOI
lat_c, dep_c, ele_c = get_voi_center(mask)
bnd = get_mask_boundary(np.transpose(mask[:, :, ele_c]))  # same contour everywhere
roi_top_mm = np.argwhere(np.transpose(mask[:, :, ele_c]).any(axis=1)).min() * sy
roi_bot_mm = np.argwhere(np.transpose(mask[:, :, ele_c]).any(axis=1)).max() * sy

mc = seg_data.motion_compensation
ext_axial = [0, nx * sx, ny * sy, 0]

ticks = [0, 20, 40, 60, 80]

# Compute baseline noise floor from the first frame of the CEUS scan
first_frame = image_data.pixel_data[:, :, :, 0]*seg_data.seg_mask
noise_floor = compute_ceus_noise_floor(first_frame)*1.5

fig, axes = plt.subplots(2, 5, figsize=(22, 9))

for col, frame in enumerate(show_frames):
    Vol = image_data.pixel_data[:, :, :, frame]

    # --- row 0: NON-COMPENSATED (raw volume) ---
    raw = np.transpose(Vol[:, :, ele_c]).astype(np.float64)
    img = enhance_ceus(raw, p_low=noise_floor, p_high_percentile=99.5)
    axes[0, col].imshow(img, cmap='gray', extent=ext_axial, aspect='equal')
    axes[0, col].axhline(roi_top_mm, color='red', ls='--', lw=1.5)
    axes[0, col].axhline(roi_bot_mm, color='red', ls='--', lw=1.5)

    # --- row 1: COMPENSATED (volume registered back) ---
    dx, dy, dz = mc.get_translation(frame)
    Vol_mc = ndshift(Vol, shift=[-dx, -dy, -dz], order=1, cval=0)
    raw_mc = np.transpose(Vol_mc[:, :, ele_c]).astype(np.float64)
    img_mc = enhance_ceus(raw_mc, p_low=noise_floor, p_high_percentile=99.5)
    axes[1, col].imshow(img_mc, cmap='gray', extent=ext_axial, aspect='equal')
    axes[1, col].axhline(roi_top_mm, color='red', ls='--', lw=1.5)
    axes[1, col].axhline(roi_bot_mm, color='red', ls='--', lw=1.5)

# Apply tick marks to all axes
for ax in axes.flat:
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.tick_params(axis='both', labelsize=16)
    

# Only show tick labels on the bottom row (x) and left column (y)
for col in range(5):
    axes[0, col].set_xticklabels([])          # hide x labels on top row
    if col != 0:
        axes[0, col].set_yticklabels([])      # hide y labels except leftmost
        axes[1, col].set_yticklabels([])

plt.tight_layout()
plt.show()

# Figure 4: Parameteric Map analysis

This script only works once finish the parametric analysis get the paramaps_LOGNORMAL. NPY files for different parameters

Figure 4: Parametric maps of 3D DCE analysis using motion compensation and without using motion compensation

In [35]:
import os
import time
import numpy as np
import napari
from skimage.morphology import remove_small_objects, binary_erosion, ball
from skimage.restoration import denoise_nl_means, estimate_sigma

# Loading the data
visualization_path = '/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/Motion_compensation_results/UCSD-P07-V03-CE1/paramaps_LOGNORMAL_mc'
AUC        = np.load(f'{visualization_path}/AUC_full_TIC_numerical.npy')   # (X, Y, Z)
image      = np.load(f'{visualization_path}/image.npy')                   # (X, Y, Z, T) or (X, Y, Z)
pixel_size = np.load(f'{visualization_path}/pix_dims.npy')                # (dx, dy, dz) mm
dx, dy, dz = pixel_size

# If we want to load the other ceus data
# image = image_data.pixel_data

time_point = 15  # initial frame the viewer opens on
# --- Load the motion-compensated VOI first so we can center the view on it ---
seg = np.load(os.path.join(visualization_path, 'segmentation.npy'))
n_frames = image.shape[-1]
mc = globals().get('seg_data', None)
mc = getattr(mc, 'motion_compensation', None) if mc is not None else None
mc_available = (mc is not None and seg.shape == seg_data.seg_mask.shape
                 and len(mc.translation_vectors) == n_frames)

if mc_available:
    voi_4d = mc.apply_to_all_frames(seg, order=0)   # (X, Y, Z, T)
else:
    print("Warning: seg_data.motion_compensation doesn't match this video "
          "(different exam or frame count) -- showing the static reference "
          "VOI only, not motion-compensated.")
    voi_4d = np.repeat(seg[..., None], n_frames, axis=-1)  # (X, Y, Z, T), unshifted

def voi_center_z(mask_3d):
    """Elevation (Z) index through the middle of the VOI, or the volume's
    geometric mid-slice if the mask is empty at this frame."""
    coords = np.argwhere(mask_3d > 0)
    if len(coords) == 0:
        return mask_3d.shape[2] // 2
    return int(round(coords[:, 2].mean()))


# --- Reorder every layer to (T, Z, X, Y) ---
ceus_volume = np.transpose(image, (3, 2, 0, 1))   # (T, Z', X, Y)
voi_4d = np.transpose(voi_4d, (3, 2, 0, 1))       # (T, Z', X, Y)
auc_3d = np.transpose(AUC, (2, 0, 1))               # (Z', X, Y), no time axis

# --- Speckle denoising, per frame ---
t0 = time.time()
ceus_denoised = np.empty_like(ceus_volume, dtype=np.float32)
for t in range(ceus_volume.shape[0]):
    frame = ceus_volume[t].astype(np.float32)                      # (Z', X, Y)
    sigma_est = float(np.mean(estimate_sigma(frame, channel_axis=0)))
    ceus_denoised[t] = denoise_nl_means(
        frame, channel_axis=0,
        h=1.15 * sigma_est, sigma=sigma_est,
        patch_size=5, patch_distance=6, fast_mode=True,
    )
    if t % 10 == 0:
        print(f"  denoised frame {t}/{ceus_volume.shape[0]}")
print(f"NLM denoised {ceus_volume.shape[0]} frames in {time.time() - t0:.1f}s")
ceus_volume = ceus_denoised

  denoised frame 0/531
  denoised frame 10/531
  denoised frame 20/531
  denoised frame 30/531
  denoised frame 40/531
  denoised frame 50/531
  denoised frame 60/531
  denoised frame 70/531
  denoised frame 80/531
  denoised frame 90/531
  denoised frame 100/531
  denoised frame 110/531
  denoised frame 120/531
  denoised frame 130/531
  denoised frame 140/531
  denoised frame 150/531
  denoised frame 160/531
  denoised frame 170/531
  denoised frame 180/531
  denoised frame 190/531
  denoised frame 200/531
  denoised frame 210/531
  denoised frame 220/531
  denoised frame 230/531
  denoised frame 240/531
  denoised frame 250/531
  denoised frame 260/531
  denoised frame 270/531
  denoised frame 280/531
  denoised frame 290/531
  denoised frame 300/531
  denoised frame 310/531
  denoised frame 320/531
  denoised frame 330/531
  denoised frame 340/531
  denoised frame 350/531
  denoised frame 360/531
  denoised frame 370/531
  denoised frame 380/531
  denoised frame 390/531
  denoised 

In [ ]:
import imageio.v3 as iio

def orient_napari_camera(viewer, angle_deg, zoom=1.0):
    theta = np.deg2rad(angle_deg)
    view_direction = (np.cos(theta), np.sin(theta), 0.0)
    up_direction = (0.0, 0.0, -1.0)
    viewer.camera.set_view_direction(view_direction, up_direction=up_direction)
    viewer.reset_view(reset_camera_angle=False)
    viewer.camera.zoom *= zoom


def _napari_sweep_angles(angle_start, angle_stop, angle_step):
    n = int(round((angle_stop - angle_start) / angle_step)) + 1
    return [angle_start + i * angle_step for i in range(n)]


def _napari_sweep_time_points(time_point):
    """Normalize time_point to a tuple: a lone int keeps the old
    single-frame behavior, an iterable (list/tuple/range/array) sweeps
    over every frame in it."""
    if isinstance(time_point, (int, np.integer)):
        return (int(time_point),)
    return tuple(int(t) for t in time_point)


def _napari_sweep_pairs(time_point, time_start, time_stop,
                         angle_start, angle_stop, angle_step):
    """(time, angle) pairs for one sweep, in two modes:

    - nested (time_start/time_stop left as None): a full angle sweep is
      captured at each frame in time_point (a single int, or an iterable
      of frame indices) -- one angle sweep per time point, back to back.
    - simultaneous (both time_start and time_stop given): time and angle
      advance together in a single pass, time interpolated across the
      same number of steps as the angle sweep -- e.g. time_start=0,
      time_stop=45 scrubs through those frames exactly as the camera
      rotates through angle_start->angle_stop, instead of a full angle
      sweep at every time point.
    """
    angles = _napari_sweep_angles(angle_start, angle_stop, angle_step)

    if time_start is not None or time_stop is not None:
        if time_start is None or time_stop is None:
            raise ValueError(
                "time_start and time_stop must both be given for a "
                "simultaneous time+angle sweep"
            )
        time_points = [int(round(t)) for t in np.linspace(time_start, time_stop, len(angles))]
        return list(zip(time_points, angles))

    time_points = _napari_sweep_time_points(time_point)
    return [(t, angle) for t in time_points for angle in angles]


def _napari_set_time(viewer, time_point):
    viewer.dims.ndisplay = 3
    current_step = list(viewer.dims.current_step)
    current_step[0] = time_point   # axis 0 = Time, per viewer.dims.axis_labels
    viewer.dims.current_step = tuple(current_step)


def napari_sweep_frames(viewer, out_dir, time_point=15, time_start=None, time_stop=None,
                         prefix="ceus_sweep", angle_start=0.0, angle_stop=90.0,
                         angle_step=5.0, scale=2, zoom=1.0):
    """Rotate the napari 3D camera and save a screenshot at each step.
    See _napari_sweep_pairs() for the nested vs. simultaneous time+angle
    modes. Mirrors save_sweep_frames() above."""
    pairs = _napari_sweep_pairs(time_point, time_start, time_stop,
                                 angle_start, angle_stop, angle_step)
    saved = []
    for t, angle in pairs:
        _napari_set_time(viewer, t)
        orient_napari_camera(viewer, angle, zoom=zoom)
        path = os.path.join(out_dir, f"{prefix}_t{t:03d}_{angle:05.1f}deg.png")
        viewer.screenshot(path, scale=scale)
        saved.append(path)
        print(f"Saved frame: {path} (t={t}, angle={angle:.1f} deg)")
    return saved


def napari_sweep_video(viewer, out_dir, filename="ceus_sweep.mp4",
                        time_point=15, time_start=None, time_stop=None,
                        angle_start=0.0, angle_stop=90.0, angle_step=5.0,
                        framerate=10, scale=2, zoom=1.0):
    """Same sweep as napari_sweep_frames(), written to an mp4 instead of
    individual PNGs. See _napari_sweep_pairs() for the nested vs.
    simultaneous time+angle modes. napari has no p.open_movie() equivalent,
    so frames are captured to in-memory arrays and encoded with imageio
    (already a napari dependency) rather than written to disk one by one."""
    pairs = _napari_sweep_pairs(time_point, time_start, time_stop,
                                 angle_start, angle_stop, angle_step)
    frames = []
    for t, angle in pairs:
        _napari_set_time(viewer, t)
        orient_napari_camera(viewer, angle, zoom=zoom)
        frames.append(viewer.screenshot(scale=scale))

    path = os.path.join(out_dir, filename)
    iio.imwrite(path, frames, fps=framerate)
    print(f"Saved video: {path} ({len(frames)} frames)")
    return path


In [37]:
raw_image = np.transpose(image, (3, 2, 0, 1))
raw_image.shape

(531, 145, 195, 144)

# Crop-to-Box (VOI) Viewer

Interactive exploration view: the full CEUS volume is still 3D-rendered (MIP/attenuated-MIP), but clipped to an axis-aligned box on all three sides -- a real crop-to-VOI, not a flat slice. Built with napari's `layer.experimental_clipping_planes` (three pairs of planes, one near/far pair per axis). Three range sliders (Z/X/Y) drive the box edges. Default box is the segmentation's own bounding box. This is a separate viewer -- it doesn't touch `viewer`/`ceus_layer` above, which the sweep-frame/video cells still depend on.

In [41]:
from magicgui import magicgui

auc_4d = mc.apply_to_all_frames(AUC, order=0)   # (X, Y, Z, T), same translation as voi_4d
auc_4d_t = np.transpose(auc_4d, (3, 2, 0, 1))   # (T, Z, X, Y)

nz_, nx_, ny_ = raw_image.shape[1:]   # (Z, X, Y) extents, for slider ranges

# Start the crop box at the full volume extent (maximum view) -- drag the
# sliders below to shrink it down to a VOI.
z_min, z_max = 0, nz_ - 1
x_min, x_max = 0, nx_ - 1
y_min, y_max = 0, ny_ - 1

viewer_crop = napari.Viewer(ndisplay=3)   # clipping planes only apply in 3D
crop_layer = viewer_crop.add_image(
    raw_image, name='CEUS', colormap='gray', gamma=1.3,
    opacity=1.0, rendering='attenuated_mip',
)

# AUC overlay -- must be added to viewer_crop, not the earlier 2D `viewer`,
# or it opens in that other window and never shows up here.
valid_vals = AUC[~np.isnan(AUC)]
auc_layer = None
if len(valid_vals) > 0:
    vmin, vmax = np.percentile(valid_vals, [5, 95])
    auc_layer = viewer_crop.add_image(
        auc_4d_t,
        name="AUC",
        colormap='turbo',
        contrast_limits=[vmin, vmax],
        opacity=0.5,
        rendering='minip',
    )

def make_box_clipping_planes(z_range, x_range, y_range):
    """Six planes (near/far per axis) that keep only the box interior."""
    (z_lo, z_hi), (x_lo, x_hi), (y_lo, y_hi) = z_range, x_range, y_range
    return [
        {'position': (z_lo, 0, 0), 'normal': (1, 0, 0), 'enabled': True},
        {'position': (z_hi, 0, 0), 'normal': (-1, 0, 0), 'enabled': True},
        {'position': (0, x_lo, 0), 'normal': (0, 1, 0), 'enabled': True},
        {'position': (0, x_hi, 0), 'normal': (0, -1, 0), 'enabled': True},
        {'position': (0, 0, y_lo), 'normal': (0, 0, 1), 'enabled': True},
        {'position': (0, 0, y_hi), 'normal': (0, 0, -1), 'enabled': True},
    ]

# Layers that stay clipped to the same crop box (CEUS + AUC, if present).
clipped_layers = [crop_layer] + ([auc_layer] if auc_layer is not None else [])
for layer in clipped_layers:
    layer.experimental_clipping_planes = make_box_clipping_planes(
        (z_min, z_max), (x_min, x_max), (y_min, y_max)
    )

# Three "bars" (range sliders) to move the box edges -- dragging either
# handle on any axis re-clips every layer in clipped_layers to the new box.
@magicgui(
    auto_call=True,
    z_range={'widget_type': 'RangeSlider', 'min': 0, 'max': nz_ - 1},
    x_range={'widget_type': 'RangeSlider', 'min': 0, 'max': nx_ - 1},
    y_range={'widget_type': 'RangeSlider', 'min': 0, 'max': ny_ - 1},
)
def set_crop_box(z_range=(z_min, z_max), x_range=(x_min, x_max), y_range=(y_min, y_max)):
    planes = make_box_clipping_planes(z_range, x_range, y_range)
    for layer in clipped_layers:
        layer.experimental_clipping_planes = planes

viewer_crop.window.add_dock_widget(set_crop_box, area='right', name='Crop box')

viewer_crop.axes.visible = True
viewer_crop.axes.labels = True
viewer_crop.axes.colored = False
# Note: bounding_box always outlines the full raw_image extent, not the
# clipped box -- it doesn't shrink as you drag the crop sliders.
crop_layer.bounding_box.visible = True
crop_layer.bounding_box.line_color = 'white'
crop_layer.bounding_box.line_thickness = 1.5
viewer_crop.dims.axis_labels = ['Time', 'Elevation (Z)', 'Lateral (X)', 'Depth (Y)']
viewer_crop.dims.current_step = (time_point, 0, 0, 0)

viewer_crop.reset_view()
viewer_crop.title = 'CEUS Cropped to VOI Bounding Box'

In [ ]:
frame_path = '/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/Motion_compensation_results/UCSD-P07-V03-CE1/paramaps_LOGNORMAL_mc/AngleFrame_MC_full_VOI'
napari_sweep_frames(viewer_crop,frame_path,time_point=38,angle_start=-45, angle_stop=45.0, angle_step=3)

In [ ]:
napari_sweep_video(viewer_crop, visualization_path, time_start=0,time_stop = 60,
                    angle_start=-45, angle_stop=45.0, angle_step=1.5)